In [ ]:
# import torch
# from peft import PeftModel
# from transformers import MBartForConditionalGeneration, MBart50Tokenizer

# ADAPTER_PATH = r"D:\khmer_spell_check\spellcheck_textsummarize\khmer-mBART50-news-summarization(2models)\khmer-mBART50-news-summarization_v2"
# BASE_MODEL = "facebook/mbart-large-50"
# SAVE_PATH = r"D:\khmer_spell_check\spellcheck_textsummarize\khmer-mBART50-news-summarization_v2_merged"

# print("Loading tokenizer...")
# tokenizer = MBart50Tokenizer.from_pretrained(
#     ADAPTER_PATH, src_lang="km_KH", tgt_lang="km_KH"
# )

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Using device:", device)

# print("Loading base model...")
# base_model = MBartForConditionalGeneration.from_pretrained(BASE_MODEL)

# print("Loading LoRA adapter...")
# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# print("Merging LoRA...")
# model = model.merge_and_unload()

# print("Saving merged model...")
# model.save_pretrained(SAVE_PATH)
# tokenizer.save_pretrained(SAVE_PATH)

# print("Merged model saved to:", SAVE_PATH)

W0404 17:44:24.465000 25184 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading tokenizer...
Using device: cuda
Loading base model...
Loading LoRA adapter...
Merging LoRA...
Saving merged model...


d:\khmer_spell_check\myvenv\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Merged model saved to: D:\khmer_spell_check\spellcheck_textsummarize\khmer-mBART50-news-summarization_v2_merged


In [1]:
import torch
from transformers import MBartForConditionalGeneration, MBart50Tokenizer

MODEL_PATH = r"D:\RAC-AI Projects\Khmer-Text-Full-System-Rith\NLP-python_service\models\khmer-mBART50-news-summarization_v2_merged"

print("Loading tokenizer...")
tokenizer = MBart50Tokenizer.from_pretrained(
    MODEL_PATH, src_lang="km_KH", tgt_lang="km_KH"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print("Loading merged model...")
model = MBartForConditionalGeneration.from_pretrained(
    MODEL_PATH, dtype=torch.float16 if device.type == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("Merged model loaded successfully!")

W0406 21:55:16.511000 45248 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading tokenizer...
Using device: cuda
Loading merged model...
Merged model loaded successfully!


In [2]:
def summarize(text: str):
    text = text.strip()
    if not text:
        return ""

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=200,
            min_length=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            forced_bos_token_id=tokenizer.lang_code_to_id["km_KH"],
            early_stopping=False,  # allow model to reach longer outputs
        )

    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    if "។" in summary:
        summary = summary[: summary.rfind("។") + 1]
    elif summary and not summary.endswith("។"):
        summary += "។"
    return f"Summary:\n{summary}"

In [3]:
khmer_text = """
អង់គ្លេស៖ ក្លិប Liverpool នឹងចូលប្រកួតប្រជែងជាមួយក្លិប Manchester United ក្នុងការដាក់សំណើតាមទិញយកខ្សែបម្រើ Jack Grealish របស់ Aston Villa ។
កីឡាករអង់គ្លេសរូបនេះ ពិតជាមានភាពលេចធ្លោ បើទោះបីជាគេមិនបានលេងឱ្យក្លិបធំៗទាំង៦នៅ Premier League ក៏ដោយ ស្របពេលកាលពីដើមរដូវកាល Grealish មានពាក្យចចាមអារ៉ាមថា បម្រុងនឹងផ្ទេរទៅចូលរួមជាមួយ Manchester United ទៅហើយ។ ពេលនេះ Liverpool កំពុងត្រូវការរុះរើ និងកសាងក្រុមឡើងវិញ បន្ទាប់ពីចាញ់ការប្រកួតក្នុងដី ដល់ទៅ៦ប្រកួតជាប់ៗគ្នា ហើយភាគីរបស់លោក Jurgen Klopp បានដាក់គោលដៅយកខ្សែបម្រើវ័យ២៥ឆ្នាំរូបនេះ មករួមក្រុម ខណៈខ្សែប្រយុទ្ធដូចជា Mohamed Salah, Sadio Mane និង Roberto Firmino ហាក់បីដូចជាធ្លាក់ទម្រង់លេងជាងរដូវកាលមុន។ Liverpool អាចទិញយក Jack Grealish បានតែត្រូវចាយប្រមាណ ៩០លានផោននៅរដូវក្ដៅខាងមុខ ប៉ុន្តែ Man UTD នៅតែជាក្លិបដែលមានប្រៀបជាងក្នុងការចរចាយក Grealish ៕
"""

print(summarize(khmer_text))


Summary:
Liverpool នឹងប្រកួតប្រជែងជាមួយ Manchester United ក្នុងការទិញយក Jack Grealish ខ្សែបម្រើ Aston Villa ពីក្លិបធំៗនៅ Premier League។ Liverpool កំពុងត្រូវការរុះរើ និងកសាងក្រុម បន្ទាប់ពីចាញ់ ៦ ប្រកួតជាប់ៗគ្នា ហើយភាគីរបស់លោក Jurgen Klopp បានដាក់គោលដៅយកខ្សែបម្រើវ័យ២៥ឆ្នាំរូបនេះ មករួមក្រុម ខណៈខ្សែប្រយុទ្ធដូចជា Mohamed Salah, Sadio Mane និង Roberto Firmino ហាក់បីដូចជាធ្លាក់ទម្រង់លេងជាងរដូវកាលមុន។ Liverpool អាចចាយប្រមាណ ៩០ លានផោននៅរដូវក្ដៅខាងមុខ ប៉ុន្តែ Man UTD នៅតែជាក្លិបដែលមានប្រៀបជាងក្នុងការចរចាយកGrealish។
